Atmospheric River / snowpack project: scan ERA5 for occurrences of ARs, warm spells, and snowmelt.  


Collect the Snotel measurements from California over the same period from USGS (there may be other sources). We would want to find periods of unusually rapid snow decrease (whether it is from melt or sublimation really does not matter). Ideally there would be some relationship between these and the "heat waves" defined from ERA5. And, for later-season snow loss, there would likely be less chance of recovery. We would explore any such relationship further.


- The U. Arizona SWE reanalysis <br>
    Daily 4 km Gridded SWE and Snow Depth from Assimilated In-Situ and Modeled Data over the Conterminous US, Version 1 <br>
    https://nsidc.org/data/nsidc-0719/versions/1
- SNODAS (from NOAA)
  Snow Data Assimilation System (SNODAS) Data Products at NSIDC, Version 1 <br>
  https://nsidc.org/data/g02158/versions/1
- possibly the UCLA snow reanalysis product <br>
  A western United States snow reanalysis dataset over the Landsat era from water years 1985 to 2021 <br>
  https://pubmed.ncbi.nlm.nih.gov/36344572/
- Jessica Lundquist evaluating these products <br>
  Snow what? Comparing snow products to reveal what works, where, and why
  https://essopenarchive.org/doi/full/10.22541/essoar.177100538.80471917/v1 


In [ ]:
Question:
SNOTEL is now the primary (and only fully-executable this pass) ground-truth melt record. Late-season 
melt-out and "recovery" happen mostly after April. What period should I pull the 36 CA SNOTEL stations' 
daily SWE for?

❯ 1. Full water year, Oct-Jul (Recommended)
     Oct 1 - Jul 31 each water year, 1990-2025 -- captures accumulation season plus full melt-out, needed to
     test the 'late-season loss doesn't recover' hypothesis.
  2. Jan-Apr only
     Match the existing blocking-events notebook window exactly, at the cost of missing most melt-out and 
     any recovery signal. 


https://www.nrcs.usda.gov/resources/data-and-reports/snow-and-climate-monitoring-predefined-reports-and-maps


#### Summary

- CA_snowmelt_events_detrend_5day.ipynb — the AR/snowpack companion of CA_z500_t700_blocking_events_detrend_5day.ipynb
  backgroud reference: Atmospheric River / snowpack project: scan ERA5 for occurrences of ARs, warm spells, and
  snowmelt. Collect the Snotel measurements from California over the same period from
  USGS (there may be other sources). We would want to find periods of unusually
  rapid snow decrease (whether it is from melt or sublimation really does not
  matter). Ideally there would be some relationship between these and the "heat
  waves" defined from ERA5. And, for later-season snow loss, there would
  likely be less chance of recovery. We would explore any such relationship
  further.

- SNOTEL+CDEC station coverage work 
    - get_snotel_box.py 
    - get_cdec_box.py,
    - qc_cdec_swe.py 
    - get_msnt_box.py → 
        - msnt_box_stations.csv (67 stations)
        - msnt_box_swe_wy1990-2026.csv (719,969 station-day records, pre-QC'd via AWDB)
    - check_msnt_cdec_overlap.py → 
        - msnt_cdec_overlap.csv (the crosswalk: for each MSNT triplet, its matching CDEC stationId)  
    - build_combined_sierra_swe.py
      fetch MSNT, crosswalk it against CDEC, merge SNTL+MSNT+CDEC into 
      combined_sierra_stations.csv / combined_sierra_swe_wy1990-2026.csv 
      (312 stations; MSNT preferred over CDEC's raw duplicate for the ~65 overlapping stations)
- Blocking-events notebook variants (_5day_new/_now/_old.ipynb) — looks like
  an in-progress comparison/consolidation
  
 SNOTEL has zero footprint south of ~38.1°N in the full Sierra box, and built get_snotel_box.py (broader box, includes NV) + get_cdec_box.py/qc_cdec_swe.py (CDEC stations covering
 Yosemite/Mammoth/Kings Canyon/Sequoia/Kern) to fill that gap.  snotel_box_stations_distribution.ipynb  

- 49 events in 1990-2025 (+3 in 2026), zero with a January onset — rapid
  network-wide decline is apparently rare before accumulation season is underway
  - Elevation and recovery are now clean, monotonic patterns across Feb→May
  (elevation-skew 0.36→0.64; recovery rate 50%→0%) — simpler than the two-humped
  story from the old Oct-Jul version, since that complexity came entirely from
  the now-excluded Oct/Jun/Jul months
  - Heat-wave match rate (Jan-Apr events only, n=32): 66% T700 / 56% Z500,
  significant at the block level (p=1.2e-06); the dry-vs-wet split is
  directionally consistent but not significant (p=0.44/0.47)
  - Top rain-on-snow case study is now May 12-16, 1996 (695% of normal implied
  rain, unmatchable against the heat-wave record since it's in May); April 7-11,
  2018 (300%) is the best matchable example
  - 2026 sets a new record on both duration and peak magnitude (23 days,
  peak_frac 0.93 — both exceed the entire 1990-2025 record)

- Added a `MIN_SWE_MM` sensitivity check (new section right after "Elevation
  representativeness" in CA_snowmelt_events_detrend_5day.ipynb): reran the pipeline at
  half (12.7mm) and double (50.8mm) the 25mm baseline. Event counts stay essentially flat
  (49 baseline vs. 51 vs. 50 in 1990-2025), the 2026 record event (Mar 10-Apr 1, 23 days)
  is identical at all three thresholds, and the Feb-to-May elevation-skew climb holds
  throughout -- 25mm is not a fragile choice.

- Added `CA_snowmelt_methodC_5daymean_std.ipynb`: snow-side analog of `CA_z500_t700_methodC_5daymean_std.ipynb`'s Method C (standard deviation of the distribution of 5-day means, instead of a >=5-consecutive-day persistence rule). Built a controlled comparison (Method A real/headline = 49 events via 5-day backward SWE diff; Method A' control = 25 events via raw 1-day diff, same persistence rule; Method C = 122/56/26 events at 1/2/3 sigma) to avoid smoothing an already-5-day-windowed quantity a second time. Verdict: Method C is a defensible, ~5x looser way to count episodes (bootstrap CIs don't overlap the control), but ranking episodes by peak sigma is NOT defensible -- January's climatological variance is ~8x smaller than April's, so modest January events (peak_frac 0.07-0.20) outrank genuinely larger April events (peak_frac 0.52-0.59) in sigma terms. Retroactively validates the original notebook's fixed-fraction (not sigma) network threshold as the right call for this variable.

- Added a `WINDOW` (backward-difference span) sensitivity check to CA_snowmelt_events_detrend_5day.ipynb, right after the MIN_SWE_MM check: swept WINDOW = 1/3/5/7 days (baseline = 5). Unlike MIN_SWE_MM, this parameter is NOT robust: event counts climb monotonically 42 (3d) -> 49 (5d) -> 55 (7d), a 31% swing, and the 2026 flagship event structurally changes at WINDOW=7 -- two separate episodes (Feb 25-onset, Mar 10-onset) merge into one continuous 36-day block. WINDOW=1 (the 'natural' day-by-day choice) is numerically broken, not just noisier: at least one station (574:CA:SNTL) has a calendar day where the pooled 36-year climatological std of the 1-day SWE diff is ~5.7e-14 (effectively zero), so standardizing divides by ~0 and corrupts sanom magnitude for that station-day (sanity check mean/std blow up to 437160/2.6e8 instead of ~0/~1). Its event count (25) looks reasonable only because find_events checks sign, not magnitude -- WINDOW=1 is flagged in the notebook as unreliable, not silently included. Takeaway: WINDOW=5 is a defensible middle point (matches CLIM_WINDOW/MIN_DURATION=5 pipeline convention) but not an independently validated optimum the way MIN_SWE_MM was.

- Added a metric x WINDOW tradeoff check to CA_snowmelt_events_detrend_5day.ipynb (right after the WINDOW sensitivity section): tested an alternative decline metric, `d5_min[t] = min(SWE over the window) - SWE[t-WINDOW]`, against the existing endpoint diff (`swe[t]-swe[t-WINDOW]`), across WINDOW=1/3/5/7. Motivation: the endpoint diff is blind to the path between its two endpoints -- a station that melts hard mid-window and then re-accumulates by day t shows '0 decline,' which hides exactly the melt-then-recovery episodes Section 6's recovery analysis cares about. The min-based metric catches those, but has its own failure mode: a single bad day (sensor glitch or same-day melt/refreeze) gets smeared across up to WINDOW days of apparent decline, since the historical dip stays inside the trailing window -- exactly what the >=5-consecutive-day persistence rule is supposed to filter out. Quantified directly with a smearing diagnostic (flag a station-day as 'stale' if its own same-day endpoint diff shows no decline at all): WINDOW=1 is numerically broken for both metrics (same 574:CA:SNTL zero-variance issue as before); min_window finds ~30-40% more events than endpoint at every surviving window (58 vs 42 at W=3, 65 vs 49 at W=5, 71 vs 55 at W=7); and min_window's smearing rate grows monotonically with WINDOW (8.1% -> 11.0% -> 13.5% of flagged station-days), while endpoint stays under 0.5% everywhere by construction. Best tradeoff if a min-based metric were adopted: WINDOW=3 (lowest smearing among numerically-sound options). Reported as a diagnostic only, not a replacement for Method A -- min_window hasn't been run through face-validity or heat-wave-matching checks, so whether its extra ~30-40% of events are real melt episodes or mostly smearing artifacts is still an open question.

- **ADOPTED the min-in-window metric (min_window, WINDOW=3) as the new Method A in CA_snowmelt_events_detrend_5day.ipynb**, replacing the original 5-day backward diff (WINDOW=5), following the stress-testing above (MIN_SWE_MM, WINDOW, and metric x WINDOW tradeoff checks). Full rebuild: Sections 2-3 now compute `d_decline = min(SWE over 3 days) - SWE at window start`; re-ran every downstream section (visualization, face validity, elevation representativeness, Z500/T700 heat-wave matching, dry/wet rain-on-snow classification, significance testing, Method C comparison, recovery analysis, 2026 out-of-sample) and rewrote every markdown conclusion with the new numbers. Headline: 58 events 1990-2025 (was 49), 4 in 2026 (was 3). Notable findings that changed, not just numbers: (1) dry-vs-wet heat-wave match rate is now statistically flat (61% vs 60% T700, p=1.00) -- previously indicative of dry matching more; (2) Method A vs Method C block-level significance flips -- Method C is now marginally MORE significant (p=7.2e-13 vs 4.7e-12) where Method A used to win; (3) Method A's raw enrichment ratio is still higher than Method C's (3.1x vs 2.7x), so the two tests now disagree on which method wins depending on which stat you ask; (4) 2026 gets a newly-detected short Apr 6-10 event the old metric missed entirely; (5) the Jan 1997 flood still isn't a detected event (3-day run, now peaking at 43-52% instead of 30-31%, same short-of-persistence conclusion). The MIN_SWE_MM sensitivity section was fixed to a LEGACY_WINDOW=5 local constant so it stays a faithful record of what was validated under the old method, decoupled from the now-changed global WINDOW.
- **Follow-up DONE**: rebuilt `CA_snowmelt_methodC_5daymean_std.ipynb`'s own "Method A (headline)" (Section 2) to reproduce the new min-in-window/WINDOW=3 metric instead of the old 5-day backward diff -- now correctly shows 58 events, matching the companion notebook. Method A' (25) and Method C (122/56/26) were untouched, since they're built from an intentionally independent 1-day-diff substrate regardless of what the companion's own headline metric is. Section 12's match rate against the real Method A shifted slightly: 55/58 (95%) matched by Method C, vs. the old 48/49 (98%) -- a marginal, within-noise decrease, not a meaningful change. Also fixed the cross-reference wording in `CA_z500_t700_methodC_5daymean_std.ipynb` that still said "already-5-day-windowed" to note the companion's window is now 3 days.

- **Added a station-precipitation cross-check for the ERA5-based dry/wet rain-on-snow classification**, prompted by the observation that this notebook already rejected ERA5 for SWE (Lundquist et al. complex-terrain bias) but relied on unchecked ERA5 precip (`tp-sf`) for the dry/wet call. Checked SNOTEL/MSNT/CDEC precipitation-sensor availability directly against the AWDB/CDEC APIs first: only 65 of 147 daily-reporting stations (44%) have any precip measurement -- all 36 SNTL (100%), 29 of 44 CDEC-D, and **zero of 67 MSNT** (that network has no precip sensor at all, only SWE) -- so a station-only replacement would go dark across nearly the whole southern Sierra; built a cross-check instead of a replacement. New fetch script `get_station_precip_box.py` (normalizes a CDEC date-format bug found along the way -- CDEC returns 'YYYY-M-D HH:MM' vs AWDB's 'YYYY-MM-DD', which silently broke pd.read_csv parse_dates). New notebook section right after the dry/wet conclusion in CA_snowmelt_events_detrend_5day.ipynb. Findings: the aggregate dry/wet signal is real and independently confirmed (Spearman rho=0.535, p=0.00001 between ERA5 implied-rain and station-measured precip; wet-classified events show significantly higher station precip than dry ones, Mann-Whitney p=0.00006) -- but ~31% (4/13) of individual 'wet' event calls don't hold up against station gauges (e.g. 2017-05-02: ERA5 says 124% of normal, stations say 1.8%), while zero 'dry' calls are contradicted by station evidence. Practical takeaway documented in the notebook: trust the aggregate dry/wet statistics, don't cite any single event's wet label as verified without checking this cross-check table.


The NWCC interactive map you linked has no network filter (networks=!), so it shows every NRCS network that reports
 snow water equivalent. In your Sierra box (34.5–40°N, roughly -121 to -116.5°W):

    ┌───────────────────┬───────┬──────────┬──────────────────────────────────┐
    │      Network      │ Code  │ In-box   │            Resolution            │
    │                   │       │  count   │                                  │
    ├───────────────────┼───────┼──────────┼──────────────────────────────────┤
    │ SNOTEL            │ SNTL  │ 36       │ Daily, automated (what the       │
    │                   │       │          │ script pulls)                    │
    ├───────────────────┼───────┼──────────┼──────────────────────────────────┤
    │ Snow              │       │          │ Semimonthly/monthly only —       │
    │ course/aerial     │ SNOW  │ 198      │ manual, some records back to     │
    │ marker            │       │          │ 1930                             │
    ├───────────────────┼───────┼──────────┼──────────────────────────────────┤
    │ Cooperator snow   │ MSNT  │ 67       │ Daily, automated (non-NRCS       │
    │ sensors           │       │          │ agencies reporting into AWDB)    │
    ├───────────────────┼───────┼──────────┼──────────────────────────────────┤
    │ SCAN              │ SCAN  │ 12       │ Daily                            │
    ├───────────────────┼───────┼──────────┼──────────────────────────────────┤
    │ SNOLITE           │ SNTLT │ 0        │ —                                │
    └───────────────────┴───────┴──────────┴──────────────────────────────────┘

looks like MSNT and SCAN 

Verified this directly against the live AWDB API
- MSNT:  "COOPERATOR SNOW SENSORS" and the exact same pipeline as SNOTEL — same
  wcc.sc.egov.usda.gov/awdbRestApi endpoint, same WTEQ element, same QC/QA flag
  vocabulary. I pulled the 67 in-box MSNT stations and they run continuously 
  from 39.9°N downto 35.97°N .
  That's precisely the Yosemite/Mammoth/Kings Canyon/Sequoia terrain the CDEC
  pull was built to cover. Spot-checked 4 stations for real data: e.g.
  CWD:CA:MSNT (Cottonwood Lakes) has 11,962 valid daily WTEQ readings spanning
  1989-10-01 to 2026-05-31 — long, dense, real records, not sparse metadata-only
  entries.

  get_msnt_box.py → msnt_box_stations.csv (67 stations),
  msnt_box_swe_wy1990-2026.csv (719,969 station-day records, pre-QC'd via AWDB)
  - check_msnt_cdec_overlap.py → msnt_cdec_overlap.csv (the crosswalk: for each
  MSNT triplet, its matching CDEC stationId). Means: MSNT adds no new geographic
  coverage beyond CDEC — it's the same 67 physical stations, just served through
  NRCS's AWDB pipeline (consistent QC/QA flagging, same format as SNOTEL)
  instead of CDEC's raw telemetry feed. The actionable split for building a
  combined dataset:- 65–67 stations: prefer the MSNT (AWDB) version over CDEC's 
  raw duplicate — better QC provenance, no need to lean on the heuristic clip/drop 
  rules in qc_cdec_swe.py. ~208 remaining CDEC stations : still the only source 
  for that coverage, keep using the QC'd CDEC feed there.
  
- SCAN — technically same reliability tier, but useless here. It is
  NRCS-operated with SNOTEL-grade QC. But I queried actual WTEQ data for all 12
  in-box SCAN stations and every single one returned zero snow data — SCAN in
  this part of the Sierra is a soil-moisture/climate network with no snow pillow
  instrumentation. 


In [ ]:
"the 500 hPa height anomaly averaged over the state is greater than one standard
  deviation"  — read literally, this computes an anomaly, averages that over the state, and compares the
  resulting single number to a standard deviation. The only standard deviation that sentence can mean is the std
  of that state-averaged series itself. That is exactly average-then-standardize (Method A) — raw field →
  spatial mean → subtract climatology of the spatial mean → divide by std of the spatial mean.

 standardize-then-average (Method B): z-score each grid
  point against its own local climatology first, then average the z-scores over the state. That's the
  alternative we'd been exploring to test whether per-point variability differences were hiding something — not
  a literal implementation of that spec's wording. And I only ran the ≥1σ threshold, not the
  repeat-for-2σ-then-3σ sweep the spec asks for.

Why this matters for the gap I found: real ridging/warm events over California aren't guaranteed to scale their absolute magnitude exactly in proportion to each location's own local variability. If a genuine synoptic event happens to produce proportionally larger physical anomalies at higher-variance interior locations (plausible — places with more volatile weather also tend to see bigger swings during an actual extreme), Method A gives that location full physical credit, while Method B systematically discounts it back down via the  division. That's a property of where the variance differs across the domain, not of how well each  is estimated — which is exactly why widening CLIM_WINDOW in the sensitivity test didn't change anything: better-estimated 's still produce the same  reweighting, just with less sampling noise riding along.